In [ ]:
# Multivariate Imputation by Chained Equations for Missing Value | MICE Algorithm | Iterative Imputer

In [25]:
import pandas as pd 
import numpy as np 

from sklearn.linear_model import LinearRegression

In [26]:
df = np.round(pd.read_csv('50_Startups.csv')[['R&D Spend', 'Administration', 'Marketing Spend', 'Profit']]/10000)
np.random.seed(9)        # Fixes the random number generator, Every time you run the code, you’ll get the same random rows.
df = df.sample(5)          # modifing the dataset to a 5 random rows dataset.
df

,R&D Spend,Administration,Marketing Spend,Profit
21,8.0,15.0,30.0,11.0
37,4.0,5.0,20.0,9.0
2,15.0,10.0,41.0,19.0
14,12.0,16.0,26.0,13.0
44,2.0,15.0,3.0,7.0


In [27]:
# since iterative imputer is applied on Input columns only, we are dropping output column(Profit) 
df = df.iloc[:, :-1]
df.head()

,R&D Spend,Administration,Marketing Spend
21,8.0,15.0,30.0
37,4.0,5.0,20.0
2,15.0,10.0,41.0
14,12.0,16.0,26.0
44,2.0,15.0,3.0


In [28]:
# we are inserting some NaN values to use iterative imputer in the dataset
df = df.copy()          # Explicitly tell pandas: “this is a copy”
df.iloc[1, 0] = np.nan 
df.iloc[3, 1] = np.nan 
df.iloc[-1, -1] = np.nan 

In [29]:
df

,R&D Spend,Administration,Marketing Spend
21,8.0,15.0,30.0
37,NaN,5.0,20.0
2,15.0,10.0,41.0
14,12.0,NaN,26.0
44,2.0,15.0,NaN


In [30]:
# Step 1 - Impute all missing values with mean of respective col 
df0 = pd.DataFrame()  

df0['R&D Spend'] = df['R&D Spend'].fillna(df['R&D Spend'].mean())
df0['Administration'] = df['Administration'].fillna(df['Administration'].mean())
df0['Marketing Spend'] = df['Marketing Spend'].fillna(df['Marketing Spend'].mean())

In [31]:
df0

,R&D Spend,Administration,Marketing Spend
21,8.00,15.00,30.00
37,9.25,5.00,20.00
2,15.00,10.00,41.00
14,12.00,11.25,26.00
44,2.00,15.00,29.25


<h2 style='color:red'>0th Iteration </h2>

<h3 style='color:blue'>Remove the col1 imputed value </h3>

In [32]:
df1 = df0.copy()
df1.iloc[1, 0] = np.nan 
df1

,R&D Spend,Administration,Marketing Spend
21,8.0,15.00,30.00
37,NaN,5.00,20.00
2,15.0,10.00,41.00
14,12.0,11.25,26.00
44,2.0,15.00,29.25


In [33]:
# Use first 3 rows to build a model and use the last for prediction 
X = df1.iloc[[0,2,3,4], 1:] 
y = df1.iloc[[0,2,3,4], 0] 

display(X) 
y

,Administration,Marketing Spend
21,15.00,30.00
2,10.00,41.00
14,11.25,26.00
44,15.00,29.25


21     8.0
2     15.0
14    12.0
44     2.0
Name: R&D Spend, dtype: float64

In [34]:
lr = LinearRegression()
lr.fit(X, y) 

lr.predict(df1.iloc[[1], 1:].values.reshape(1,2))          # 1 sample × 2 features  →  shape = (1, 2)

C:\Users\rupes\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


array([23.14158651])

In [35]:
df1.iloc[1,0] = 23.14

In [36]:
df1

,R&D Spend,Administration,Marketing Spend
21,8.00,15.00,30.00
37,23.14,5.00,20.00
2,15.00,10.00,41.00
14,12.00,11.25,26.00
44,2.00,15.00,29.25


<h3 style='color:blue'>Remove the col2 imputed value </h3>

In [37]:
df1.iloc[3, 1] = np.nan 
df1

,R&D Spend,Administration,Marketing Spend
21,8.00,15.0,30.00
37,23.14,5.0,20.00
2,15.00,10.0,41.00
14,12.00,NaN,26.00
44,2.00,15.0,29.25


In [38]:
# Use last 3 rows to build a model and use the first for prediction 
X = df1.iloc[[0,1,2,4], [0,2]]
y = df1.iloc[[0,1,2,4], 1]

lr = LinearRegression()
lr.fit(X, y) 
lr.predict(df.iloc[[3], [0,2]].values.reshape(1,2))

C:\Users\rupes\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


array([11.06331285])

In [39]:
df1.iloc[3, 1] = 11.06

In [40]:
df1

,R&D Spend,Administration,Marketing Spend
21,8.00,15.00,30.00
37,23.14,5.00,20.00
2,15.00,10.00,41.00
14,12.00,11.06,26.00
44,2.00,15.00,29.25


<h3 style='color:blue'>Remove the col3 imputed value </h3>

In [41]:
df1.iloc[-1,-1] = np.nan 
df1

,R&D Spend,Administration,Marketing Spend
21,8.00,15.00,30.0
37,23.14,5.00,20.0
2,15.00,10.00,41.0
14,12.00,11.06,26.0
44,2.00,15.00,NaN


In [42]:
# Use last 3 rows to build a model and use the first for prediction 
X = df1.iloc[:-1, :-1]
y = df1.iloc[:-1, -1]

lr = LinearRegression()
lr.fit(X, y) 
lr.predict(df.iloc[-1, :-1].values.reshape(1,2))

C:\Users\rupes\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


array([31.56351448])

In [43]:
df1.iloc[-1, -1] = 31.56

In [44]:
# After 1st Iteration 
df1

,R&D Spend,Administration,Marketing Spend
21,8.00,15.00,30.00
37,23.14,5.00,20.00
2,15.00,10.00,41.00
14,12.00,11.06,26.00
44,2.00,15.00,31.56


In [45]:
# Subtract 0th iteration from 1st iteration
df1 - df0

,R&D Spend,Administration,Marketing Spend
21,0.00,0.00,0.00
37,13.89,0.00,0.00
2,0.00,0.00,0.00
14,0.00,-0.19,0.00
44,0.00,0.00,2.31


In [46]:
# repeat this same steps and do a number of iteration(10-20) to reach near to true value.